# Structured Outputs: JSON, Schema Validation, Constrained Decoding Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: JSON Schema Validator

Build a validator from scratch that checks whether a Python object matches a JSON Schema. This is what runs on the output side to verify compliance.

In [ ]:
```python

import json

def validate_schema(data, schema):

    errors = []

    _validate(data, schema, "", errors)

    return errors

def _validate(data, schema, path, errors):

    schema_type = schema.get("type")

    if schema_type == "object":

        if not isinstance(data, dict):

            errors.append(f"{path}: expected object, got {type(data).__name__}")

            return

        for key in schema.get("required", []):

            if key not in data:

                errors.append(f"{path}.{key}: required field missing")

        properties = schema.get("properties", {})

        for key, value in data.items():

            if key in properties:

                _validate(value, properties[key], f"{path}.{key}", errors)

    elif schema_type == "array":

        if not isinstance(data, list):

            errors.append(f"{path}: expected array, got {type(data).__name__}")

            return

        min_items = schema.get("minItems", 0)

        max_items = schema.get("maxItems", float("inf"))

        if len(data) < min_items:

            errors.append(f"{path}: array has {len(data)} items, minimum is {min_items}")

        if len(data) > max_items:

            errors.append(f"{path}: array has {len(data)} items, maximum is {max_items}")

        items_schema = schema.get("items", {})

        for i, item in enumerate(data):

            _validate(item, items_schema, f"{path}[{i}]", errors)

    elif schema_type == "string":

        if not isinstance(data, str):

            errors.append(f"{path}: expected string, got {type(data).__name__}")

            return

        enum_values = schema.get("enum")

        if enum_values and data not in enum_values:

            errors.append(f"{path}: '{data}' not in allowed values {enum_values}")

    elif schema_type == "number":

        if not isinstance(data, (int, float)):

            errors.append(f"{path}: expected number, got {type(data).__name__}")

            return

        minimum = schema.get("minimum")

        maximum = schema.get("maximum")

        if minimum is not None and data < minimum:

            errors.append(f"{path}: {data} is less than minimum {minimum}")

        if maximum is not None and data > maximum:

            errors.append(f"{path}: {data} is greater than maximum {maximum}")

    elif schema_type == "boolean":

        if not isinstance(data, bool):

            errors.append(f"{path}: expected boolean, got {type(data).__name__}")

    elif schema_type == "integer":

        if not isinstance(data, int) or isinstance(data, bool):

            errors.append(f"{path}: expected integer, got {type(data).__name__}")

In [ ]:
```

### Step 2: Pydantic-Style Model to Schema

Build a minimal class-to-schema converter. Define a Python class and generate its JSON Schema automatically.

In [ ]:
```python

class SchemaField:

    def __init__(self, field_type, required=True, default=None, enum=None, minimum=None, maximum=None):

        self.field_type = field_type

        self.required = required

        self.default = default

        self.enum = enum

        self.minimum = minimum

        self.maximum = maximum

def python_type_to_schema(field):

    type_map = {

        str: "string",

        int: "integer",

        float: "number",

        bool: "boolean",

    }

    schema = {}

    if field.field_type in type_map:

        schema["type"] = type_map[field.field_type]

    elif field.field_type == list:

        schema["type"] = "array"

        schema["items"] = {"type": "string"}

    elif isinstance(field.field_type, dict):

        schema = field.field_type

    if field.enum:

        schema["enum"] = field.enum

    if field.minimum is not None:

        schema["minimum"] = field.minimum

    if field.maximum is not None:

        schema["maximum"] = field.maximum

    return schema

def model_to_schema(name, fields):

    properties = {}

    required = []

    for field_name, field in fields.items():

        properties[field_name] = python_type_to_schema(field)

        if field.required:

            required.append(field_name)

    return {

        "type": "object",

        "properties": properties,

        "required": required,

    }

In [ ]:
```

### Step 3: Constrained Token Filter

Simulate constrained decoding. Given a partial JSON string and a schema, determine which token categories are valid at the current position.

In [ ]:
```python

def next_valid_tokens(partial_json, schema):

    stripped = partial_json.strip()

    if not stripped:

        return ["{"]

    try:

        json.loads(stripped)

        return ["<EOS>"]

    except json.JSONDecodeError:

        pass

    last_char = stripped[-1] if stripped else ""

    if last_char == "{":

        return ['"', "}"]

    elif last_char == '"':

        if stripped.endswith('":'):

            return ['"', "0-9", "true", "false", "null", "[", "{"]

        return ["a-z", '"']

    elif last_char == ":":

        return [" ", '"', "0-9", "true", "false", "null", "[", "{"]

    elif last_char == ",":

        return [" ", '"', "{", "["]

    elif last_char in "0123456789":

        return ["0-9", ".", ",", "}", "]"]

    elif last_char == "}":

        return [",", "}", "]", "<EOS>"]

    elif last_char == "]":

        return [",", "}", "<EOS>"]

    elif last_char == "[":

        return ['"', "0-9", "true", "false", "null", "{", "[", "]"]

    else:

        return ["any"]

def demonstrate_constrained_decoding():

    partial_states = [

        '',

        '{',

        '{"product"',

        '{"product":',

        '{"product": "Sony"',

        '{"product": "Sony",',

        '{"product": "Sony", "price":',

        '{"product": "Sony", "price": 348',

        '{"product": "Sony", "price": 348}',

    ]

    print(f"{'Partial JSON':<45} {'Valid Next Tokens'}")

    print("-" * 80)

    for state in partial_states:

        valid = next_valid_tokens(state, {})

        display = state if state else "(empty)"

        print(f"{display:<45} {valid}")

In [ ]:
```

### Step 4: Extraction Pipeline

Combine everything into an extraction pipeline: define a schema, simulate an LLM producing structured output, validate the output, and handle retries.

In [ ]:
```python

def simulate_llm_extraction(text, schema, attempt=0):

    if "headphones" in text.lower() or "sony" in text.lower():

        if attempt == 0:

            return '{"product": "Sony WH-1000XM5", "price": 348.00, "in_stock": true, "categories": ["audio", "headphones"]}'

        return '{"product": "Sony WH-1000XM5", "price": 348.00, "in_stock": true}'

    if "laptop" in text.lower():

        return '{"product": "MacBook Pro 16", "price": 2499.00, "in_stock": false, "categories": ["computers"]}'

    return '{"product": "Unknown", "price": 0, "in_stock": false}'

def extract_with_retry(text, schema, max_retries=3):

    for attempt in range(max_retries):

        raw = simulate_llm_extraction(text, schema, attempt)

        try:

            data = json.loads(raw)

        except json.JSONDecodeError as e:

            print(f"  Attempt {attempt + 1}: JSON parse error -- {e}")

            continue

        errors = validate_schema(data, schema)

        if not errors:

            return data

        print(f"  Attempt {attempt + 1}: Schema validation errors -- {errors}")

    return None

product_schema = {

    "type": "object",

    "properties": {

        "product": {"type": "string"},

        "price": {"type": "number", "minimum": 0},

        "in_stock": {"type": "boolean"},

        "categories": {"type": "array", "items": {"type": "string"}},

    },

    "required": ["product", "price", "in_stock"],

}

In [ ]:
```

### Step 5: Run the Full Pipeline

In [ ]:
```python

def run_demo():

    print("=" * 60)

    print("  Structured Output Pipeline Demo")

    print("=" * 60)

    print("\n--- Schema Definition ---")

    product_fields = {

        "product": SchemaField(str),

        "price": SchemaField(float, minimum=0),

        "in_stock": SchemaField(bool),

        "categories": SchemaField(list, required=False),

    }

    generated_schema = model_to_schema("Product", product_fields)

    print(json.dumps(generated_schema, indent=2))

    print("\n--- Schema Validation ---")

    test_cases = [

        ({"product": "Test", "price": 10.0, "in_stock": True}, "Valid object"),

        ({"product": "Test", "price": -5.0, "in_stock": True}, "Negative price"),

        ({"product": "Test", "in_stock": True}, "Missing price"),

        ({"product": "Test", "price": "ten", "in_stock": True}, "String as price"),

        ("not an object", "String instead of object"),

    ]

    for data, label in test_cases:

        errors = validate_schema(data, product_schema)

        status = "PASS" if not errors else f"FAIL: {errors}"

        print(f"  {label}: {status}")

    print("\n--- Constrained Decoding Simulation ---")

    demonstrate_constrained_decoding()

    print("\n--- Extraction Pipeline ---")

    texts = [

        "The Sony WH-1000XM5 headphones are priced at $348 and currently available.",

        "The new MacBook Pro 16-inch laptop costs $2499 but is sold out.",

        "This is a random sentence with no product info.",

    ]

    for text in texts:

        print(f"\n  Input: {text[:60]}...")

        result = extract_with_retry(text, product_schema)

        if result:

            print(f"  Output: {json.dumps(result)}")

        else:

            print(f"  Output: FAILED after retries")

In [ ]:
```

## Exercises

In [ ]:
1. Extend the schema validator to support `oneOf` (the data must match exactly one of several schemas). This handles polymorphic outputs -- for example, a field that can be either a `Product` or a `Service` object with different shapes.

2. Build a "schema diff" tool that compares two schemas and identifies breaking changes (removed required fields, changed types) versus non-breaking changes (added optional fields, relaxed constraints). This is essential for versioning your extraction schemas in production.

3. Implement a more realistic constrained decoding simulator. Given a JSON Schema and a vocabulary of 100 tokens (letters, digits, punctuation, keywords), walk through generation step by step, masking invalid tokens at each position. Measure what percentage of the vocabulary is valid at each step.

4. Build an extraction eval suite. Create 50 product descriptions with hand-labeled JSON outputs. Run your extraction pipeline on all 50 and measure exact match, field-level accuracy, and type compliance. Identify which fields are hardest to extract correctly.

5. Add "confidence scores" to your extraction pipeline. For each extracted field, estimate how confident the model is (based on token probabilities, or by running extraction 3 times and measuring consistency). Flag low-confidence fields for human review.